# 供应链运营数据分析

本 Notebook 展示从数据加载、指标计算、风险识别到行动建议输出的完整分析流程。示例数据均为虚构数据。

In [ ]:
from pathlib import Path

from src.data_processing import load_data
from src.analysis import (
    abc_classification,
    analyze_inventory,
    build_action_list,
    calculate_kpis,
    enrich_orders,
    supplier_performance,
)
from src.visualization import generate_all_charts

orders, inventory = load_data()
print(f'订单记录：{len(orders)}，库存 SKU：{len(inventory)}')

## 1. 核心 KPI

In [ ]:
kpis = calculate_kpis(orders, inventory)
kpis

## 2. 交付与库存风险

In [ ]:
order_detail = enrich_orders(orders)
inventory_health = analyze_inventory(inventory)

order_detail[[
    'order_id', 'supplier_id', 'lead_time_days', 'delay_days',
    'on_time_flag', 'order_value', 'defect_rate'
]].head()

In [ ]:
inventory_health[[
    'product_id', 'product_name', 'available_stock', 'days_of_supply',
    'risk_level', 'risk_reason', 'recommended_order_qty'
]].sort_values(['risk_level', 'days_of_supply']).reset_index(drop=True)

## 3. ABC 分类与供应商绩效

In [ ]:
abc = abc_classification(orders)
suppliers = supplier_performance(orders)

display(abc)
display(suppliers)

## 4. 风险行动清单

In [ ]:
actions = build_action_list(inventory_health, abc)
actions

## 5. 生成图表

In [ ]:
chart_paths = generate_all_charts(
    inventory_health, suppliers, abc, Path('outputs')
)
chart_paths